**What is RAG?**
RAG stands for Retrieval Augmented Generation.

It was introduced in the paper Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.

Each step can be roughly broken down to:

**Retrieval**- Seeking relevant information from a source given a query. For example, getting relevant passages of Wikipedia text from a database given a question.

**Augmented** - Using the relevant retrieved information to modify an input to a generative model (e.g. an LLM).

**Generation**- Generating an output given an input. For example, in the case of an LLM, generating a passage of text given an input prompt

**Why RAG?**
The main goal of RAG is to improve the generation outptus of LLMs.

**Two primary improvements can be seen as:**

**Preventing hallucinations** - LLMs are incredible but they are prone to potential hallucination, as in, generating something that looks correct but isn't. RAG pipelines can help LLMs generate more factual outputs by providing them with factual (retrieved) inputs. And even if the generated answer from a RAG pipeline doesn't seem correct, because of retrieval, you also have access to the sources where it came from.

**Work with custom data**- Many base LLMs are trained with internet-scale text data. This means they have a great ability to model language, however, they often lack specific knowledge. RAG systems can provide LLMs with domain-specific data such as medical information or company documentation and thus customized their outputs to suit specific use cases.
RAG can also be a much quicker solution to implement than fine-tuning an LLM on specific data.

In [1]:
# =====================================================================
# 🚀 LIGHTWEIGHT RAG SYSTEM — MiniLM + (Gemini or an open-source LLM) + FAISS
# =====================================================================
#
# TUTORIAL NOTE: This notebook is designed to be run top-to-bottom once,
# then interacted with entirely through the Gradio app launched in the
# final cell. Nobody's API key is stored in this notebook, and the app
# lets each user pick between two REMOTE LLM APIs — nothing is ever
# downloaded or hosted inside this Colab session:
#   • Gemini 2.5 Flash — via Google's API, needs the user's own free
#     Google API key.
#   • Qwen2.5-7B-Instruct — via the Hugging Face Inference API, needs
#     the user's own free Hugging Face access token. This is a plain
#     network call to Hugging Face's hosted servers (routed to the
#     "together" provider), just like the Gemini call is a network call
#     to Google's servers — no model weights are ever downloaded or
#     loaded into this notebook's memory.
#
# WHY QWEN INSTEAD OF MISTRAL: as of this writing, Hugging Face's own
# model page for every current Mistral model (Mistral-7B-Instruct-v0.3,
# Magistral, Mistral-Medium, etc.) states "This model isn't deployed by
# any Inference Provider" — Mistral AI runs its own separate API instead,
# so it simply cannot be reached this way. Qwen2.5-7B-Instruct is an
# ungated, Apache-2.0-licensed, similarly-sized open model that IS
# actively deployed on Hugging Face's Inference Providers network, so it
# was swapped in as a like-for-like open-source option.
# So nobody is blocked just because they couldn't get a Google API key
# working, and nobody has to wait on a multi-gigabyte model download.

# ================================
# STEP 0 — Install Dependencies
# ================================
# All installs live in this one cell so there's a single place to look
# if something is missing, and so we don't reinstall the same packages
# multiple times later in the notebook.
!pip install --quiet --upgrade langchain langchain-community faiss-cpu pypdf sentence-transformers
!pip install --quiet --upgrade openai langchain-huggingface python-docx google-generativeai openpyxl
!pip install --quiet --upgrade "gradio>=5.0" transformers huggingface_hub

# NOTE: If you already ran an older version of this notebook earlier in this
# same Colab session (i.e. gradio/transformers were imported before this
# install ran), the upgrade won't take effect until you restart the runtime:
#   Runtime -> Restart session, then Runtime -> Run all
# Do this once if needed, then you shouldn't need to do it again.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requ

In [2]:
# ================================
# STEP 1 — Environment Setup (no keys here!)
# ================================
# This notebook intentionally does NOT configure any API key or token at
# import time, and there is no hardcoded key anywhere in this file. Each
# user supplies their OWN credential at runtime in the app's Setup tab,
# for whichever provider they pick:
#   • Gemini      -> their own Google API key   (https://aistudio.google.com/apikey)
#   • Open-source -> their own Hugging Face access token
#                    (https://huggingface.co/settings/tokens — a free "Read" token is enough)
# Either credential is kept only in memory for that session.
import os
import torch
import google.generativeai as genai

print("✅ Environment ready. Choose Gemini or the open-source model in the app's Setup tab below.")

✅ Environment ready. Choose Gemini or the open-source model in the app's Setup tab below.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# ==========================================
# STEP 2 — Import Required Libraries
# ==========================================
import pandas as pd
import requests
from bs4 import BeautifulSoup

from sentence_transformers import SentenceTransformer
from docx import Document
from pypdf import PdfReader

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# Create Project Folder Structure
# ==========================================
os.makedirs("rag_data/chunks", exist_ok=True)
os.makedirs("rag_data/vector_store", exist_ok=True)

/tmp/ipykernel_723/2843192207.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
# ==========================================================
# STEP 3 — Loaders for Different File Types
# ==========================================================
# These functions just extract plain text from each source type. The app's
# Setup tab (last cell) calls these when a user uploads files — there is no
# separate ingestion step to run manually in the notebook anymore.

def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text


def load_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs])


def load_excel(path):
    df = pd.read_excel(path)
    return df.to_string()


def scrape_url(url):
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "header", "footer", "nav", "aside", "noscript"]):
        tag.extract()

    texts = []
    for h in soup.find_all(['h1', 'h2', 'h3', 'h4']):
        t = h.get_text(strip=True)
        if len(t) > 10:
            texts.append(t)
    for p in soup.find_all("p"):
        t = p.get_text(strip=True)
        if len(t) > 20:
            texts.append(t)
    for li in soup.find_all("li"):
        t = li.get_text(strip=True)
        if len(t) > 20:
            texts.append(t)

    return "\n".join(texts)

## STEP 4 — Embedding Model & Knowledge Base State
Unlike the earlier version of this notebook, there's no `input()` prompt here asking you to upload files or paste a URL. Uploading documents (PDF / DOCX / Excel) now happens **once, inside the Gradio app itself**, on the Setup tab — and so does the choice of which LLM answers questions. This cell loads the (small, local) embedding model used for retrieval and sets up the shared state object the rest of the notebook — and the app — will use. Note: the *embedding* model and the cross-encoder re-ranker below still run locally (they're tiny and fast); it's only the big generative LLM — Gemini or the open-source model — that's called remotely.

In [5]:
# ==========================================================
# STEP 4 — Embedding Model & Vector Store Helpers
# ==========================================================
from langchain_huggingface import HuggingFaceEmbeddings

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model ready.")


class RAGState:
    """Single place holding everything about the current session's knowledge
    base and LLM configuration, instead of scattering `global` variables
    across the notebook.

    Note: because the Gradio app below launches with `share=True`, this state
    is shared by anyone who opens the public link while your Colab runtime is
    alive. That's fine for a personal demo/tutorial session, but don't use
    this pattern as-is for a multi-user production app — give each user their
    own `gr.State` in that case.
    """
    def __init__(self):
        self.provider = None          # "gemini" or "hf_oss"
        # Gemini (remote API) fields
        self.api_key = None
        self.gemini_model = None
        # Open-source model (remote API via Hugging Face Inference Providers) fields
        self.hf_token = None
        self.hf_client = None
        # Shared knowledge-base fields
        self.all_text = ""
        self.vector_db = None
        self.retriever = None
        self.chat_history = []
        self.initialized = False


state = RAGState()


def update_vector_db():
    """(Re)builds the FAISS index from everything ingested so far."""
    if not state.all_text.strip():
        return "⚠️ No text found to index."

    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
    chunks = splitter.split_text(state.all_text)

    state.vector_db = FAISS.from_texts(texts=chunks, embedding=embedding_function)
    state.retriever = state.vector_db.as_retriever(search_kwargs={"k": 20})
    return f"📦 Vector store built — {len(chunks)} chunks indexed."


def configure_gemini(api_key):
    """Configures the Gemini client with a user-supplied key and does a
    lightweight sanity check before committing to it. Returns (success, message).
    The key itself is never printed, logged, or written to disk."""
    if not api_key or not api_key.strip():
        return False, "❌ Please enter a Google API key (or switch to the open-source option)."

    api_key = api_key.strip()
    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-2.5-flash")
        # Cheap call to confirm the key actually works before we commit to it.
        model.generate_content(
            "ping",
            generation_config=genai.types.GenerationConfig(max_output_tokens=5),
        )
    except Exception as e:
        return False, f"❌ Google rejected this key: {e}"

    state.api_key = api_key
    state.gemini_model = model
    # "Session secret": lives only in this runtime's environment variables —
    # never written to disk, never saved into this notebook file, and wiped
    # the moment the Colab runtime restarts or disconnects.
    os.environ["GOOGLE_API_KEY"] = api_key
    return True, "✅ Gemini API key verified and saved for this session."


⏳ Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready.


### STEP 4B — Optional: Open-Source LLM via the Hugging Face Inference API (remote, no download)
For anyone who can't get a Google API key working (corporate network blocks, no Google account, quota issues, etc.), the app also offers **`Qwen/Qwen2.5-7B-Instruct`** — reached through the **Hugging Face Inference API**, a hosted, serverless endpoint (routed to the `together` provider) running on infrastructure Hugging Face already partners with.

This is architecturally identical to how Gemini is used above: every request is a network call to someone else's servers using the user's own credential. **Nothing is downloaded, nothing is loaded into this notebook's memory or GPU, and there's no multi-gigabyte wait** — the only thing needed is a free Hugging Face access token from `https://huggingface.co/settings/tokens` (a "Read" token is enough).

*(Why not Mistral? As of this writing, Hugging Face's own model pages for every current Mistral model — including `Mistral-7B-Instruct-v0.3` — say "This model isn't deployed by any Inference Provider." Mistral AI runs its own separate hosted API rather than being available through Hugging Face's provider network, so it can't be called this way. Qwen2.5-7B-Instruct is a similarly-sized, ungated, Apache-2.0 open model that IS actively deployed, so it's used here as the open-source option.)*

In [6]:
# ==========================================================
# STEP 4B — Remote LLM Helper: Open-source model via Hugging Face Inference API
# ==========================================================
# No model weights are downloaded or loaded here. `InferenceClient` just
# opens an HTTPS connection to Hugging Face's hosted inference servers,
# exactly the way `genai` opens one to Google's servers for Gemini.
from huggingface_hub import InferenceClient

# Qwen2.5-7B-Instruct is ungated (Apache 2.0) and, unlike the Mistral models,
# is actively deployed on Hugging Face's Inference Providers network (via
# the "together" provider), so it works out of the box with just a free HF token.
OPEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OPEN_MODEL_PROVIDER = "together"


def configure_open_model(hf_token):
    """Configures a remote Hugging Face Inference API client for the
    open-source model and does a lightweight sanity check before committing
    to it. Returns (success, message). The token itself is never printed,
    logged, or written to disk."""
    if not hf_token or not hf_token.strip():
        return False, "❌ Please enter a Hugging Face access token (or switch to the Gemini option)."

    hf_token = hf_token.strip()
    try:
        client = InferenceClient(model=OPEN_MODEL_ID, provider=OPEN_MODEL_PROVIDER, token=hf_token)
        # Cheap call to confirm the token + model access actually work before we commit to it.
        client.chat_completion(
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=5,
        )
    except Exception as e:
        return False, f"❌ Hugging Face rejected this request: {e}"

    state.hf_token = hf_token
    state.hf_client = client
    # "Session secret": lives only in this runtime's environment variables —
    # never written to disk, never saved into this notebook file, and wiped
    # the moment the Colab runtime restarts or disconnects.
    os.environ["HF_TOKEN"] = hf_token
    return True, f"✅ Hugging Face token verified — {OPEN_MODEL_ID} ready via remote API (nothing downloaded)."


def generate_with_open_model(prompt, max_new_tokens=768):
    """One turn of generation via the remote Hugging Face Inference API."""
    response = state.hf_client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


print(f"✅ Remote open-source LLM helper ready ({OPEN_MODEL_ID} via Hugging Face — nothing downloaded).")

✅ Remote open-source LLM helper ready (Qwen/Qwen2.5-7B-Instruct via Hugging Face — nothing downloaded).


### STEP 5 — Cross-Encoder Re-ranking

While FAISS (Bi-Encoders) is extremely fast at finding candidate chunks, it sometimes misses the specific semantic relationship between a query and a document.

A **Cross-Encoder** processes the query and the document chunk simultaneously, providing a much higher accuracy relevance score. We will use this to re-rank the initial candidates fetched by FAISS. (This model is small — ~80MB — so it loads locally in seconds; it's only the big generative LLM that's kept remote.)

In [7]:
from sentence_transformers import CrossEncoder

# Initialize a pre-trained Cross-Encoder model trained for passage re-ranking
print("⏳ Loading Cross-Encoder Re-ranker (ms-marco-MiniLM)...")
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Re-ranker Loaded!")

def rerank_chunks(query, documents, top_n=5):
    """
    Scores a list of LangChain Document objects against a query using a Cross-Encoder.
    Returns the top_n most relevant documents.
    """
    if not documents:
        return []

    # Prepare the input pairs for the Cross-Encoder: [[query, text1], [query, text2], ...]
    doc_texts = [doc.page_content for doc in documents]
    pairs = [[query, text] for text in doc_texts]

    # Get relevance scores (higher is better)
    scores = rerank_model.predict(pairs)

    # Pair scores with original documents and sort by score descending
    scored_docs = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)

    # Extract and return only the top_n documents
    return [doc for score, doc in scored_docs[:top_n]]

⏳ Loading Cross-Encoder Re-ranker (ms-marco-MiniLM)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Re-ranker Loaded!


## STEP 6 — Finalized RAG Logic
This is the single, final version of the RAG function: retrieval, re-ranking, and conversational memory in one place. It reads `state.provider` to decide whether to answer via the Gemini API or the Hugging Face Inference API for the open-source model — both remote calls, no local generation — and reads the vector index off the shared `state` object set up by the Gradio app's Setup tab. It simply refuses politely if setup hasn't run yet.

In [8]:
import google.generativeai as genai

# =====================================================================
# STEP 6 — Production-Ready RAG Logic (provider-agnostic: Gemini or the
# open-source model, both reached via remote API calls — no local hosting)
# =====================================================================

def rag_chat_v2(query):
    if not state.initialized:
        return ("⚠️ Please finish Setup first: pick Gemini or the open-source model, upload your "
                "documents, and click **Initialize RAG System** — then come back to this tab.")

    # 1. Broad Retrieval (FAISS) — pull 20 candidates for the re-ranker
    initial_docs = state.retriever.vectorstore.similarity_search(query, k=20)

    # 2. Semantic Re-ranking (Cross-Encoder) — narrow down to the top 7
    best_docs = rerank_chunks(query, initial_docs, top_n=7)

    # 3. Context Preparation
    context = "\n---\n".join([d.page_content for d in best_docs])

    # 4. Memory Integration — last 4 turns for continuity
    history_text = "\n".join([f"User: {u}\nAI: {a}" for u, a in state.chat_history[-4:]])

    # 5. Prompt Construction
    prompt = f"""
You are a professional RAG assistant. Answer the question based on the provided context and history.
If the answer isn't in the context, state that clearly.

[CONTEXT]
{context}

[CONVERSATION HISTORY]
{history_text}

[USER QUESTION]
{query}

[DETAILED RESPONSE]
"""

    # 6. Generation — a single remote API call, routed to whichever provider was set up
    if state.provider == "gemini":
        response = state.gemini_model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=2048,
                temperature=0.3
            )
        )
        answer = response.text if response.text else "I'm sorry, I couldn't generate a response based on the data. Please try rephrasing."

    elif state.provider == "hf_oss":
        answer = generate_with_open_model(prompt) or "I'm sorry, I couldn't generate a response based on the data. Please try rephrasing."

    else:
        return "⚠️ No LLM provider is configured yet. Please complete Setup first."

    state.chat_history.append((query, answer))
    return answer

print("✅ RAG logic ready — it will activate once Setup finishes in the app below.")

✅ RAG logic ready — it will activate once Setup finishes in the app below.


## 🚀 Final Step: Launch Professional Gradio RAG Interface

This is the **launch point** for the whole tutorial. Everything happens here:

1. **Setup tab** — choose **Gemini 2.5 Flash** (paste your own free Google API key) *or* **Qwen2.5-7B-Instruct** (paste your own free Hugging Face access token). Both are remote API calls — nothing is downloaded, and there's no waiting on a model to load. Then upload your knowledge base (PDF / DOCX / Excel) and click **Initialize RAG System** once.
2. **Chat & Voice tab** — ask questions by typing or by speaking (local Whisper transcribes your voice for free, no credential needed for that part either).

No API key, token, or document is baked into this notebook — everything is supplied by whoever runs the app, and anyone without a working Google API key can still complete the whole tutorial using their own Hugging Face token instead.

In [11]:
import gradio as gr
from transformers import pipeline
import torch
import os

print("Gradio version:", gr.__version__)

# --- Open Source STT Model Initialization ---
print("⏳ Loading Open Source STT Model (Whisper Base)...")
stt_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("✅ STT Model Loaded!")

def transcribe_audio(audio_path):
    # Check if audio_path exists and is not None
    if audio_path is None:
        return gr.update()

    try:
        print(f"Transcribing: {audio_path}")
        result = stt_pipe(audio_path)
        text = result["text"].strip()
        return text if text else "[No speech detected]"
    except Exception as e:
        return f"[Transcription Error: {str(e)}]"

LOADERS = {".pdf": load_pdf, ".docx": load_docx, ".xlsx": load_excel, ".xls": load_excel}

GEMINI_OPTION = "Gemini 2.5 Flash (needs a free Google API key)"
OPEN_MODEL_OPTION = f"{OPEN_MODEL_ID} — open-source (needs a free Hugging Face access token)"

def toggle_credential_fields(model_choice):
    is_gemini = (model_choice == GEMINI_OPTION)
    return gr.update(visible=is_gemini), gr.update(visible=not is_gemini)

def initialize_system(model_choice, api_key, hf_token, files):
    if model_choice == GEMINI_OPTION:
        ok, provider_msg = configure_gemini(api_key)
        if ok: state.provider = "gemini"
    else:
        ok, provider_msg = configure_open_model(hf_token)
        if ok: state.provider = "hf_oss"

    if not ok:
        return provider_msg, gr.update(interactive=False)

    if not files:
        return provider_msg + "\n❌ Please upload at least one document.", gr.update(interactive=False)

    processed, skipped = [], []
    for f in files:
        ext = os.path.splitext(f.name)[1].lower()
        loader = LOADERS.get(ext)
        if not loader:
            skipped.append(f"{os.path.basename(f.name)} (unsupported type)")
            continue
        try:
            state.all_text += loader(f.name) + "\n"
            processed.append(os.path.basename(f.name))
        except Exception as e:
            skipped.append(f"{os.path.basename(f.name)} ({e})")

    index_msg = update_vector_db()
    state.initialized = state.retriever is not None

    status_lines = [provider_msg]
    if processed: status_lines.append(f"✅ Processed: {', '.join(processed)}")
    if skipped: status_lines.append(f"⚠️ Skipped: {', '.join(skipped)}")
    status_lines.append(index_msg)
    status_lines.append("🎉 Ready! Switch to the 'Chat & Voice' tab.")

    return "\n".join(status_lines), gr.update(interactive=state.initialized)

def gradio_chat(message, history):
    if not message or message.strip() == "":
        return "", history
    try:
        response_text = rag_chat_v2(message)
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response_text})
        return "", history
    except Exception as e:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": f"System Error: {str(e)}"})
        return "", history

CUSTOM_CSS = """
.gradio-container { max-width: 1100px !important; margin: 0 auto !important; }
.setup_card, .chat_card { border-radius: 16px !important; box-shadow: 0 2px 14px rgba(0,0,0,0.06) !important; }
footer {display: none !important;}
"""

theme = gr.themes.Soft(primary_hue="emerald", secondary_hue="slate")

with gr.Blocks(title="Professional RAG Assistant") as demo:
    gr.Markdown("# 🌿 Professional RAG Assistant")

    with gr.Tab("⚙️ Setup"):
        model_choice = gr.Radio(choices=[GEMINI_OPTION, OPEN_MODEL_OPTION], value=GEMINI_OPTION, label="LLM Provider")
        api_key_box = gr.Textbox(label="Google API Key", type="password", visible=True)
        hf_token_box = gr.Textbox(label="Hugging Face Token", type="password", visible=False)
        file_box = gr.File(label="Documents", file_count="multiple")
        init_btn = gr.Button("🚀 Initialize RAG System", variant="primary")
        status_box = gr.Textbox(label="Status", interactive=False, lines=4)

        model_choice.change(toggle_credential_fields, inputs=model_choice, outputs=[api_key_box, hf_token_box])

    with gr.Tab("💬 Chat & Voice"):
        with gr.Row():
            with gr.Column(scale=4):
                chatbot = gr.Chatbot(height=480)
                msg = gr.Textbox(placeholder="Type your question...", label="Your message")
                send_btn = gr.Button("Send", variant="primary", interactive=False)

            with gr.Column(scale=1):
                gr.Markdown("### 🎤 Voice input")
                audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Record & Stop")
                gr.Markdown("_Click 'Stop' to transcribe._")

        audio_input.stop_recording(fn=transcribe_audio, inputs=[audio_input], outputs=[msg])
        audio_input.change(fn=transcribe_audio, inputs=[audio_input], outputs=[msg])
        send_btn.click(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
        msg.submit(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])

    init_btn.click(initialize_system, inputs=[model_choice, api_key_box, hf_token_box, file_box], outputs=[status_box, send_btn])

demo.launch(debug=True, share=True, theme=theme, css=CUSTOM_CSS)

Gradio version: 6.23.1
⏳ Loading Open Source STT Model (Whisper Base)...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

✅ STT Model Loaded!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1ba4985bc279228e2f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Transcribing: /tmp/gradio/ca85d8a0611c10b9ce64712179257f80d5190270236e254e26ace7e9c393bc28/audio.wav
Transcribing: /tmp/gradio/ca85d8a0611c10b9ce64712179257f80d5190270236e254e26ace7e9c393bc28/audio.wav
Transcribing: /tmp/gradio/f3f160643db143c5c96935e434148ae64e989db340be578e4f69d6ccdea877fc/audio.wav
Transcribing: /tmp/gradio/f3f160643db143c5c96935e434148ae64e989db340be578e4f69d6ccdea877fc/audio.wav
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1ba4985bc279228e2f.gradio.live
